# 03. Feature Engineering
**Responsibility:** Vaibhav - feature engineering and training-only scaling.

**Goal:** Build the 29 inputs and scale them using only legitimate training rows, so nothing leaks from validation or test data.

**Input:** the three CSV files saved by notebook 02.

**Output:** `data/processed/features.npz`, `models/scaler.joblib` and `results/feature_columns.json`.

Start this notebook in a fresh Python session. If the split files are missing, run notebook 02 first.

### Files used in this notebook
- **Read from notebook 02:** `data/processed/train.csv`, `data/processed/validation.csv`, `data/processed/test.csv`.
- **Save for notebooks 04 and 05:** `data/processed/features.npz` — five named numerical arrays, explained in the save section.
- **Save for reuse/evidence:** `models/scaler.joblib` and `results/feature_columns.json`.

No new model is trained here. This notebook only prepares the inputs.

## Setup for this notebook
Run from the project root or the `notebooks` folder. The next cell finds the root folder, so saved paths work in either case. Each notebook loads the previous stage's saved files. It does not depend on variables left in another notebook's Python session.

`Path` handles folders, `os.chdir` selects the project root, `pd` handles tables, `np` handles numbers, and `plt` draws charts. The `display` and `Image` tools show tables and pictures.

In [1]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "requirements.txt").exists(), "Open this notebook from the project folder."
os.chdir(ROOT)
Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

from sklearn.preprocessing import StandardScaler
import joblib

## 1. Read the saved splits
`float_precision="round_trip"` preserves the saved floating-point values when reading the CSV. This avoids small changes caused only by saving and loading.

The checks confirm chronological order and both classes in the evaluation periods. `assert` stops execution if a required check fails.

**Files in the next cell:** `data/processed/test.csv`, `data/processed/train.csv`, `data/processed/validation.csv`.

`read_csv`, `np.load` and `joblib.load` read files. `to_csv`, `write_text`, `savefig`, `np.savez_compressed` and `joblib.dump` save files. The file map explains what each file contains.

In [2]:
assert Path("data/processed/train.csv").exists(), "Run notebook 02 first."
train = pd.read_csv("data/processed/train.csv", float_precision="round_trip")
validation = pd.read_csv("data/processed/validation.csv", float_precision="round_trip")
test = pd.read_csv("data/processed/test.csv", float_precision="round_trip")
assert train["Time"].max() < validation["Time"].min()
assert validation["Time"].max() < test["Time"].min()
assert validation["Class"].nunique() == 2 and test["Class"].nunique() == 2
print("Split files loaded in time order.")

Split files loaded in time order.


## 2. Prepare the 29 inputs
**What we do:** Create LogAmount, keep V1-V28, and select only legitimate training rows.

**Why:** We want the GMM to learn legitimate behavior. We keep Class out of the input because it is the answer we later check. Time has already done its job of ordering the periods.

The list comprehension creates the names V1, V2, up to V28. Adding `["LogAmount"]` gives **29 input columns**. In pandas, `table[features]` selects those columns, while `table[condition]` selects rows.

In [3]:
train["LogAmount"] = np.log1p(train["Amount"])
validation["LogAmount"] = np.log1p(validation["Amount"])
test["LogAmount"] = np.log1p(test["Amount"])

features = ["V" + str(i) for i in range(1, 29)] + ["LogAmount"]
normal_train = train[train["Class"] == 0].copy()

X_train = normal_train[features]
X_validation = validation[features]
X_test = test[features]
y_test = test["Class"].to_numpy()

assert "Class" not in features and "Time" not in features
print("Training:", X_train.shape, " Validation:", X_validation.shape, " Test:", X_test.shape)

Training: (169893, 29)  Validation: (56745, 29)  Test: (56746, 29)


**How to read the output:** `(169893, 29)` means 169,893 legitimate training transactions, each described by 29 features. The 342 training frauds do not enter GMM fitting.

`X` means inputs. `y` means correct answers. `to_numpy()` turns the test-label column into a numerical array for later calculations.

We do use labels to select known legitimate training examples. The GMM itself receives only X, with no fraud targets. Calling the whole preparation completely label-free would be incorrect.

## 3. Scale the inputs
**What we do:** Use StandardScaler to put inputs on a common numerical scale.

**Why:** Columns can have different centers and spreads. Scaling gives the model more consistent numerical units.

For each column: **scaled value = (original value - training mean) / training standard deviation**.

`fit_transform` learns these numbers and applies them. `transform` applies the already learned numbers. We fit only on legitimate training rows. Fitting on future test data would let future information influence the model; this is called **data leakage**.

**Small example:** If a feature has training mean 100 and standard deviation 50, a value of 150 becomes `(150 - 100) / 50 = 1`. It is one training standard deviation above the mean. These are teaching numbers, not estimates from our data.

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print("First three scaled inputs for the first training transaction:")
print(X_train_scaled[0, :3])
assert np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-10)

First three scaled inputs for the first training transaction:
[-0.67390624 -0.06370234  1.58778996]


**How to read the output:** A value near 0 is near that feature's legitimate training average. A negative value is below it; a positive value is above it. A negative scaled value does not mean a negative purchase amount.

The last line checks that the transformed training columns are centered near zero. Scaling parameters stay fixed for validation and test.

## 4. Save the inputs and scaler
An `.npz` file is a container holding named numerical arrays. The next notebook reads the arrays by name. `np.savez_compressed` saves disk space without changing their values.

We save the scaler separately so the exact same learned mean and standard deviation can be reused. The model inputs have 29 columns. Label arrays are stored separately and never passed to `GMM.fit`.

These files come from this local run. Load only model bundles you generated or otherwise trust.

| Name inside `features.npz` | What it contains | Read by |
|---|---|---|
| `X_train` | Scaled legitimate training inputs | 04 |
| `X_validation` | Scaled mixed validation inputs | 05 |
| `X_test` | Scaled mixed test inputs | 05 |
| `y_validation` | Separate validation labels | 05, to select legitimate validation scores |
| `y_test` | Separate test labels | 05, to measure mistakes |

The saved names are shorter than the in-memory names: for example `X_train=X_train_scaled` saves the scaled array under the name `X_train`. No further scaling happens in 04 or 05.

**Files in the next cell:** `data/processed/features.npz`, `models/scaler.joblib`, `results/feature_columns.json`.

`read_csv`, `np.load` and `joblib.load` read files. `to_csv`, `write_text`, `savefig`, `np.savez_compressed` and `joblib.dump` save files. The file map explains what each file contains.

In [5]:
Path("models").mkdir(exist_ok=True)
np.savez_compressed("data/processed/features.npz",
    X_train=X_train_scaled, X_validation=X_validation_scaled, X_test=X_test_scaled,
    y_validation=validation["Class"].to_numpy(), y_test=y_test)
joblib.dump(scaler, "models/scaler.joblib")
Path("results/feature_columns.json").write_text(json.dumps(features, indent=2))
assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_validation_scaled).all() and np.isfinite(X_test_scaled).all()
assert int(scaler.n_samples_seen_) == len(normal_train)
print("Saved", len(normal_train), "legitimate training rows with", len(features), "inputs.")

Saved 169893 legitimate training rows with 29 inputs.


**Checkpoint:** Training has 169,893 rows and 29 inputs. Class and Time are absent from those inputs. Only legitimate training rows fitted the scaler.

**Next:** `04_GMM_Training.ipynb`.